# RF modulation classification

Classify HisarMod2019.1 snippets into 26 modulation types in five families.
Each float32 example has shape `(2, 1024)`: in-phase (I) and quadrature (Q).
SNR is metadata in dB, never a model input.

Run cells in order using the project environment. Section 9 now includes CNN
training through ordinary notebook cells. Prepare the data on Amarel before
running the loading and training cells. Run All starts training. Feature
extraction, comparison plots, and final test evaluation remain planned.
See README.md for setup.

## 1. Imports

NumPy handles signal arrays, pandas groups accuracy by noise level,
scikit-learn creates the stratified split, and PyTorch defines the CNN.
All imports are collected here so later cells share the same namespace.

In [ ]:
import json
import random
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.ticker import PercentFormatter
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset

## 2. Modulation labels and families

Keep the dataset's raw codes throughout preparation and reporting.
The final digit (`code % 10`) identifies the family. For example, code 21
means 16QAM and belongs to family 1 (QAM). Dictionary order groups related
modulations together for eventual confusion matrices.

In [ ]:
# Raw label codes from the dataset readme, grouped by family (code % 10) so
# confusion matrices show within-family mistakes as blocks on the diagonal.
MODULATION_NAMES = {
    0: "BPSK",
    10: "QPSK",
    20: "8PSK",
    30: "16PSK",
    40: "32PSK",
    50: "64PSK",
    1: "4QAM",
    11: "8QAM",
    21: "16QAM",
    31: "32QAM",
    41: "64QAM",
    51: "128QAM",
    61: "256QAM",
    2: "2FSK",
    12: "4FSK",
    22: "8FSK",
    32: "16FSK",
    3: "4PAM",
    13: "8PAM",
    23: "16PAM",
    4: "AM-DSB",
    14: "AM-DSB-SC",
    24: "AM-USB",
    34: "AM-LSB",
    44: "FM",
    54: "PM",
}

# Family numbers are code % 10.
FAMILY_NAMES = {
    0: "PSK",
    1: "QAM",
    2: "FSK",
    3: "PAM",
    4: "analog",
}

## 3. Load converted data

Run `scripts/convert_hisarmod.py` once on Amarel through its existing Slurm
script. It converts MATLAB complex-number CSV text to `.npy` arrays.
Keeping this separate avoids repeating the expensive conversion during learning.

`load_partition` accepts the data directory and either `train` or `test`.
It returns signals `(n, 2, 1024)`, raw labels `(n,)`, and SNRs `(n,)` in dB.
Memory mapping lets the full signal array stay on disk until rows are accessed.
Use the planned fixed subset locally; use full arrays on Amarel.
Data inspection and subset creation remain to be implemented.

In [ ]:
def load_partition(data_dir, partition):
    iq = np.load(data_dir / f"{partition}_iq.npy", mmap_mode="r")
    labels = np.load(data_dir / f"{partition}_labels.npy")
    snr_db = np.load(data_dir / f"{partition}_snr_db.npy")
    return iq, labels, snr_db

## 4. Normalize each signal's power

For complex samples `I + jQ`, instantaneous power is `I**2 + Q**2`.
Divide each signal by the square root of its own average power to obtain
unit average power. No statistics are shared across examples.

Apply this helper to individual examples or batches. Normalizing the entire
memory-mapped dataset at once would allocate large arrays in memory.

In [ ]:
def normalize_power(iq):
    sample_power = np.sum(iq**2, axis=-2, keepdims=True)
    average_power = np.mean(sample_power, axis=-1, keepdims=True)
    normalized = iq / np.sqrt(average_power)
    return normalized.astype(np.float32, copy=False)

## 5. Create one shared training/validation split

Reserve the authors' test set for the final evaluation. Split their training
set 80/20, preserving the proportions of every (modulation, SNR) pair.
Use the same fixed seed for both models so comparisons use identical examples.

This helper returns row indices rather than copies of the signals. The indices
are sorted for disk access; the training DataLoader shuffles them,
because source rows are ordered by SNR and modulation.

In [ ]:
def split_train_validation(labels, snr_db, seed):
    train_rows, validation_rows = train_test_split(
        np.arange(len(labels)),
        test_size=0.2,
        stratify=np.column_stack((labels, snr_db)),
        random_state=seed,
    )
    return np.sort(train_rows), np.sort(validation_rows)

## 6. Measure accuracy by SNR

An overall score hides how performance changes with noise. This helper compares
predictions to true labels and averages correctness separately at each SNR.
Pass raw codes for modulation accuracy, or both code arrays modulo 10 for
family accuracy. Both models will use this same function on validation data.
The next helper adds overall accuracy; confusion matrices appear after validation prediction collection.

In [ ]:
def accuracy_by_snr(true_codes, predicted_codes, snr_db):
    correct = pd.Series(true_codes == predicted_codes, name="accuracy")
    return correct.groupby(pd.Index(snr_db, name="snr_db")).mean()

### Calculate shared metrics

Compare raw codes to measure modulation accuracy, and compare `code % 10`
to measure family accuracy. A wrong modulation within the correct family
counts as a family success. All accuracies are fractions between 0 and 1.

This function accepts aligned, nonempty one-dimensional arrays, independent
of the model. It returns overall accuracies and a table indexed by SNR in dB.
The existing helper calculates each SNR group's accuracy separately.

In [ ]:
def calculate_metrics(true_codes, predicted_codes, snr_db):
    overall = {
        "modulation_accuracy": float(np.mean(true_codes == predicted_codes)),
        "family_accuracy": float(
            np.mean(true_codes % 10 == predicted_codes % 10)
        ),
    }
    by_snr = pd.DataFrame(
        {
            "modulation_accuracy": accuracy_by_snr(
                true_codes, predicted_codes, snr_db
            ),
            "family_accuracy": accuracy_by_snr(
                true_codes % 10, predicted_codes % 10, snr_db
            ),
        }
    )
    return overall, by_snr

## 7. Map raw codes to CNN class indices

Cross-entropy uses contiguous target indices from 0 to 25. These mappings
translate at the model boundary while preserving raw codes elsewhere.
`CODE_TO_INDEX` maps raw codes to indices; `CLASS_CODES` maps indices back
to raw codes. Their order follows the modulation dictionary above.

In [ ]:
# The CNN scores classes 0-25; the data uses raw codes (0, 10, ..., 54).
# Convert between them only here, at the model boundary.
CLASS_CODES = list(MODULATION_NAMES)
CODE_TO_INDEX = {code: index for index, code in enumerate(CLASS_CODES)}

## 8. Define the small 1D CNN

Each convolution learns local patterns along the time axis. Batch normalization
and ReLU follow each convolution, and max pooling halves the sequence length.
The four blocks increase the feature channels from 2 to 32, 64, 128, and 256.
Global average pooling reduces each feature channel to one value; dropout and
a linear layer produce 26 class scores (logits).

The class defines the architecture only. Instantiating, training, and running
predictions are later steps. The existing comments show the tensor shapes
through the network. Cross-entropy will consume logits directly.

In [ ]:
class ModulationCNN(nn.Module):
    def __init__(self, n_classes=26, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            # (batch, 2, 1024) -> (batch, 32, 512)
            nn.Conv1d(2, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            # -> (batch, 64, 256)
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            # -> (batch, 128, 128)
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            # -> (batch, 256, 64)
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.MaxPool1d(2),
            # -> (batch, 256)
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(256, n_classes),
        )

    def forward(self, iq):
        features = self.features(iq)
        logits = self.classifier(features)
        return logits

## 9. Train the CNN

After converting the data on Amarel, set `training_params.data_dir` to the converted
arrays and run the cells in order. Training uses only the authors' training
partition, split 80/20 with the fixed seed. No test-set data is loaded.
For a shorter run, point `training_params.data_dir` at a prepared real-data subset and reduce
`training_params.epochs`. There is no synthetic data mode.

The initial settings below use up to 50 epochs and stop after five epochs
without lower validation loss. Tune these settings on validation data only.

Edit the `TrainingParameters` defaults below or override them when creating
`training_params`, for example `TrainingParameters(epochs=10, batch_size=64)`.

In [ ]:
@dataclass
class TrainingParameters:
    data_dir: Path = Path("data/processed")
    seed: int = 42
    batch_size: int = 128
    learning_rate: float = 0.0003
    dropout: float = 0.3
    epochs: int = 50
    patience: int = 10


training_params = TrainingParameters()

### Read examples one at a time

The Dataset holds the memory-mapped array and row indices, not a copied split.
Each access normalizes one signal and converts its raw label to a class index.
Training shuffles examples each epoch; validation preserves order. Use zero
loader workers for a simple notebook workflow without subprocess pickling.

In [ ]:
class SignalDataset(Dataset):
    def __init__(self, iq, labels, rows):
        self.iq = iq
        self.labels = labels
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        signal = torch.from_numpy(normalize_power(self.iq[row]))
        return signal, CODE_TO_INDEX[int(self.labels[row])]

### Train for one epoch

This function has one job: update model weights using each training batch.
`train()` enables dropout and batch-normalization updates. Clear gradients,
compute logits and cross-entropy, backpropagate, and take an Adam step.
Return loss averaged over examples, so the last partial batch has the right
weight. This loss is measured while the weights are changing.

In [ ]:
def run_training_epoch(model, device, train_loader, optimizer):
    model.train()
    total_loss = 0.0
    total_examples = 0
    for signals, targets in train_loader:
        signals, targets = signals.to(device), targets.to(device)
        optimizer.zero_grad()
        logits = model(signals)
        loss = nn.functional.cross_entropy(logits, targets)
        torch.autograd.backward(loss)
        optimizer.step()
        total_loss += loss.item() * len(targets)
        total_examples += len(targets)
    return total_loss / total_examples

### Evaluate without updating weights

`eval()` disables dropout and stops batch-normalization updates. `no_grad()`
skips gradient recording. Sum cross-entropy across examples and count correct
class predictions, then divide by the number of examples. This function
returns mean loss and modulation accuracy; the training loop uses it only
on validation data. Family and SNR breakdowns remain separate work.

In [ ]:
def evaluate(model, device, data_loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    with torch.no_grad():
        for signals, targets in data_loader:
            signals, targets = signals.to(device), targets.to(device)
            logits = model(signals)
            total_loss += nn.functional.cross_entropy(
                logits, targets, reduction="sum"
            ).item()
            total_correct += int((logits.argmax(dim=1) == targets).sum().item())
            total_examples += len(targets)
    return total_loss / total_examples, total_correct / total_examples

### Put the epochs together

Create a fresh CNN and Adam optimizer, then call the two functions above for
each epoch. Record training loss, validation loss, and validation accuracy.
Save weights whenever validation loss improves; stop after `patience` epochs
without improvement. Finally restore the best weights and return the model
in evaluation mode together with its history.

Checkpoint weights are saved during training. JSON settings and result files
are handled in separate cells below. History is saved after training returns;
an interrupted run retains its best checkpoint but may lose its loss history.

In [ ]:
def train(
    training_params, device, train_loader, validation_loader, checkpoint_path
):
    if training_params.epochs < 1 or training_params.patience < 1:
        raise ValueError("epochs and patience must be at least one")
    model = ModulationCNN(dropout=training_params.dropout).to(device)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=training_params.learning_rate
    )
    history = []
    best_loss = float("inf")
    epochs_without_improvement = 0

    for epoch in range(1, training_params.epochs + 1):
        training_loss = run_training_epoch(
            model, device, train_loader, optimizer
        )
        validation_loss, validation_accuracy = evaluate(
            model, device, validation_loader
        )
        history.append(
            {
                "epoch": epoch,
                "train_loss": training_loss,
                "validation_loss": validation_loss,
                "validation_accuracy": validation_accuracy,
            }
        )
        print(
            f"Epoch {epoch}: train {training_loss:.4f}, "
            f"val {validation_loss:.4f}, accuracy {validation_accuracy:.3f}"
        )
        if validation_loss < best_loss:
            best_loss = validation_loss
            epochs_without_improvement = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= training_params.patience:
            print("Early stopping.")
            break

    model.load_state_dict(
        torch.load(checkpoint_path, map_location=device, weights_only=True)
    )
    model.eval()
    return model, history

### Prepare data and choose the device

Seed Python, NumPy, and PyTorch before creating the model. A separate seeded
generator shuffles training batches. Validation batches stay in order.
Use CUDA when available, otherwise CPU. GPU results can still vary slightly.

In [ ]:
random.seed(training_params.seed)
np.random.seed(training_params.seed)
torch.manual_seed(training_params.seed)
torch.cuda.manual_seed_all(training_params.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
iq, labels, snr_db = load_partition(training_params.data_dir, "train")
train_rows, validation_rows = split_train_validation(
    labels, snr_db, training_params.seed
)
train_loader = DataLoader(
    SignalDataset(iq, labels, train_rows),
    batch_size=training_params.batch_size,
    shuffle=True,
    generator=torch.Generator().manual_seed(training_params.seed),
    num_workers=0,
)
validation_loader = DataLoader(
    SignalDataset(iq, labels, validation_rows),
    batch_size=training_params.batch_size,
    shuffle=False,
    num_workers=0,
)
print(
    f"Device: {device}; train: {len(train_rows)}; val: {len(validation_rows)}"
)

### Record the settings

Use the existing `runs/` and `models/` folders in the project root.
Save settings to `runs/config.json`, epoch history to `runs/metrics.json`,
and the best model weights to `models/best.pt`. Rerunning these cells
replaces the previous files; copy any results you want to keep first.
On a fresh checkout or Amarel, create both folders before running the notebook
with `mkdir -p runs models`. Run the notebook from the project root.

Create `runs/figures/` here for the seven PNG plots. Validation predictions
are saved together as an NPZ file, overall metrics as JSON, and per-SNR
accuracies as CSV. These reporting files are also replaced when rerun.

In [ ]:
run_dir = Path("runs")
model_dir = Path("models")
figures_dir = run_dir / "figures"
figures_dir.mkdir(exist_ok=True)
checkpoint_path = model_dir / "best.pt"
config = {
    **asdict(training_params),
    "data_dir": str(training_params.data_dir),
    "device": str(device),
    "class_codes": CLASS_CODES,
}
(run_dir / "config.json").write_text(json.dumps(config, indent=2) + "\n")

### Run training

The setup above defines the data, device, and output paths. This call trains
the CNN and returns its best weights and epoch history. Running this cell
starts training. **Run All also loads the data and starts training.**

In [ ]:
model, history = train(
    training_params, device, train_loader, validation_loader, checkpoint_path
)

### Save the results

The model already holds the best validation weights, also saved at
`checkpoint_path`. Save the returned history for later learning curves.
The checkpoint contains weights only, not optimizer state for resuming.

In [ ]:
(run_dir / "metrics.json").write_text(json.dumps(history, indent=2) + "\n")
print(f"Best weights: {checkpoint_path}; metrics: {run_dir / 'metrics.json'}")

### Training and validation loss

Plot the mean cross-entropy recorded at each completed epoch. Training loss
is measured while weights change and dropout is active; validation loss is
measured afterward in evaluation mode. Validation loss can therefore be lower
than training loss. Falling training loss with rising validation loss can
indicate overfitting.

In [ ]:
history_df = pd.DataFrame(history)

plt.figure(figsize=(6, 4))
plt.plot(history_df["epoch"], history_df["train_loss"], label="Training")
plt.plot(history_df["epoch"], history_df["validation_loss"], label="Validation")

plt.title("Training and validation loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(
    figures_dir / "training_validation_loss.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

### Collect predictions in validation order

Use the restored best model to predict every validation example. `eval()`
disables dropout and batch-normalization updates; `no_grad()` disables gradient
tracking. Choose the highest-scoring class, move each batch to CPU, and join
its predictions into one array. Convert class indices back to raw label codes.

The validation loader has `shuffle=False` and follows `validation_rows`, so
`true_codes`, `predicted_codes`, and `validation_snr_db` are aligned arrays of
shape `(n_validation,)`. SNR is in dB and is never passed to the model.
These arrays will feed the shared reports in the next step.

Save the three aligned arrays together in `runs/validation_predictions.npz`
so reporting can be repeated without running inference again.

In [ ]:
model.eval()
prediction_batches = []

with torch.no_grad():
    for signals, _ in validation_loader:
        signals = signals.to(device)
        logits = model(signals)
        predicted_indices = logits.argmax(dim=1)
        prediction_batches.append(predicted_indices.cpu().numpy())

predicted_indices = np.concatenate(prediction_batches)

# Convert CNN class indices back to the dataset's raw modulation codes.
class_codes = np.array(CLASS_CODES, dtype=np.int64)
predicted_codes = class_codes[predicted_indices]

# These follow the same row order as the validation loader.
true_codes = labels[validation_rows]
validation_snr_db = snr_db[validation_rows]

np.savez(
    run_dir / "validation_predictions.npz",
    true_codes=true_codes,
    predicted_codes=predicted_codes,
    snr_db=validation_snr_db,
)

### Display validation metrics

Use the aligned predictions collected above. Overall accuracy weights every
validation example equally; the table shows accuracy separately at each SNR.
The same function can later report the baseline's predictions.

Save overall accuracies as JSON and the per-SNR table as CSV, including its
SNR index. Each plotting cell saves its figure before displaying it.

In [ ]:
overall_metrics, metrics_by_snr = calculate_metrics(
    true_codes, predicted_codes, validation_snr_db
)
(run_dir / "validation_metrics.json").write_text(
    json.dumps(overall_metrics, indent=2) + "\n"
)
metrics_by_snr.to_csv(run_dir / "validation_accuracy_by_snr.csv")

print(pd.Series(overall_metrics).to_string())
metrics_by_snr

### Modulation accuracy versus SNR

Exact modulation accuracy requires the predicted raw code to match the true
code. Each point shows validation accuracy at one SNR level.
Both figures use a 0–100% scale for comparison.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(
    metrics_by_snr.index,
    metrics_by_snr["modulation_accuracy"],
    marker="o",
    label="CNN",
)
ax.set_title("Modulation accuracy")
ax.set_xlabel("SNR (dB)")
ax.set_ylabel("Validation accuracy")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.savefig(
    figures_dir / "modulation_accuracy_by_snr.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

### Family accuracy versus SNR

Family accuracy compares the families derived from the raw codes. A prediction
can have the correct family even when its exact modulation is wrong.
Both figures use a 0–100% scale for comparison.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(
    metrics_by_snr.index,
    metrics_by_snr["family_accuracy"],
    marker="o",
    label="CNN",
)
ax.set_title("Family accuracy")
ax.set_xlabel("SNR (dB)")
ax.set_ylabel("Validation accuracy")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.savefig(
    figures_dir / "family_accuracy_by_snr.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

### Confusion matrices at low and high SNR

Compare validation examples below 0 dB with those at or above 0 dB.
Rows are true classes and columns are predictions. Each row is normalized
by the number of examples in its true class; 0.80 means 80%. All four figures
use color only and share the same 0–1 scale. Labels stay in family order.
A class absent from a subset has a zero row, not measured zero accuracy.

Each figure has its own plotting cell. These cells assume both SNR groups
contain examples, as expected for the full stratified validation split.

In [ ]:
low_snr = validation_snr_db < 0
high_snr = validation_snr_db >= 0

### Validation families — SNR < 0 dB

In [ ]:
matrix = confusion_matrix(
    true_codes[low_snr] % 10,
    predicted_codes[low_snr] % 10,
    labels=list(FAMILY_NAMES),
    normalize="true",
)

plt.figure(figsize=(6, 6))
plt.imshow(matrix, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="Fraction of true class")
plt.xticks(range(5), list(FAMILY_NAMES.values()), rotation=90)
plt.yticks(range(5), list(FAMILY_NAMES.values()))
plt.xlabel("Predicted family")
plt.ylabel("True family")
plt.title("Validation families — SNR < 0 dB")
plt.tight_layout()
plt.savefig(
    figures_dir / "family_confusion_low_snr.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

### Validation families — SNR ≥ 0 dB

In [ ]:
matrix = confusion_matrix(
    true_codes[high_snr] % 10,
    predicted_codes[high_snr] % 10,
    labels=list(FAMILY_NAMES),
    normalize="true",
)

plt.figure(figsize=(6, 6))
plt.imshow(matrix, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="Fraction of true class")
plt.xticks(range(5), list(FAMILY_NAMES.values()), rotation=90)
plt.yticks(range(5), list(FAMILY_NAMES.values()))
plt.xlabel("Predicted family")
plt.ylabel("True family")
plt.title("Validation families — SNR ≥ 0 dB")
plt.tight_layout()
plt.savefig(
    figures_dir / "family_confusion_high_snr.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

### Validation modulations — SNR < 0 dB

In [ ]:
matrix = confusion_matrix(
    true_codes[low_snr],
    predicted_codes[low_snr],
    labels=list(MODULATION_NAMES),
    normalize="true",
)

plt.figure(figsize=(12, 12))
plt.imshow(matrix, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="Fraction of true class")
plt.xticks(range(26), list(MODULATION_NAMES.values()), rotation=90)
plt.yticks(range(26), list(MODULATION_NAMES.values()))
plt.xlabel("Predicted modulation")
plt.ylabel("True modulation")
plt.title("Validation modulations — SNR < 0 dB")
plt.tight_layout()
plt.savefig(
    figures_dir / "modulation_confusion_low_snr.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

### Validation modulations — SNR ≥ 0 dB

In [ ]:
matrix = confusion_matrix(
    true_codes[high_snr],
    predicted_codes[high_snr],
    labels=list(MODULATION_NAMES),
    normalize="true",
)

plt.figure(figsize=(12, 12))
plt.imshow(matrix, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="Fraction of true class")
plt.xticks(range(26), list(MODULATION_NAMES.values()), rotation=90)
plt.yticks(range(26), list(MODULATION_NAMES.values()))
plt.xlabel("Predicted modulation")
plt.ylabel("True modulation")
plt.title("Validation modulations — SNR ≥ 0 dB")
plt.tight_layout()
plt.savefig(
    figures_dir / "modulation_confusion_high_snr.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()

## 10. Build the feature baseline — planned

Explain and calculate cumulants (C20, C21, C40, C41, C42) and amplitude/phase
statistics, then fit a HistGradientBoostingClassifier. Any transformation fit
across examples must be fit on training data only. Use the same split and
validation metrics as the CNN.

## 11. Compare, evaluate once, and interpret — planned

Compare both models on validation data and freeze their configurations.
Only then explicitly enable the final evaluation on the authors' test set.
Keep that evaluation disabled during routine notebook execution.

Report overall accuracy, accuracy versus SNR, and confusion matrices for both
26 modulations and five families. Derive family predictions from modulation
predictions rather than training another model. Explain the main confusions
and the limitations of simulated data. See plan.md for the remaining work.